In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from Bio import SeqIO
from Bio.PDB import PDBParser
from Bio.PDB.SASA import ShrakeRupley

OUT = Path("results/structure")
OUT.mkdir(parents=True, exist_ok=True)
FUNCTIONAL = (300, 324, 325, 328, 347, 410)
MAX_ASA = {
    "A": 129.0, "R": 274.0, "N": 195.0, "D": 193.0, "C": 167.0, "Q": 225.0,
    "E": 223.0, "G": 104.0, "H": 224.0, "I": 197.0, "L": 201.0, "K": 236.0,
    "M": 224.0, "F": 240.0, "P": 159.0, "S": 155.0, "T": 172.0, "W": 285.0,
    "Y": 263.0, "V": 174.0,
}
THREE = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLN": "Q",
    "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I", "LEU": "L", "LYS": "K",
    "MET": "M", "PHE": "F", "PRO": "P", "SER": "S", "THR": "T", "TRP": "W",
    "TYR": "Y", "VAL": "V",
}
RANGES = (
    (25, 48, "none"),
    (49, 236, "2.60.40.1730"),
    (237, 333, "3.30.2010.30"),
    (334, 488, "1.10.390.10"),
    (489, 629, "1.25.40.320"),
)


def domain_of(position):
    for start, end, ident in RANGES:
        if start <= position <= end:
            return ident
    raise SystemExit(f"position {position} is outside the mature protein")


def heavy_coords(residue):
    coords = []
    for atom in residue:
        element = (atom.element or atom.get_name()[0]).strip().upper()
        if element != "H":
            coords.append(atom.coord)
    return np.asarray(coords, dtype=float)


def representative_atom(residue):
    if residue.resname == "GLY":
        return residue["CA"].coord.astype(float)
    if "CB" in residue:
        return residue["CB"].coord.astype(float)
    return residue["CA"].coord.astype(float)


records = list(SeqIO.parse("results/alignment/colap_trimmed_aligned.fasta", "fasta"))
reference = next(record for record in records if record.id.startswith("GCF_000012325.1|"))
column_of = {}
position = 24
for column, residue in enumerate(str(reference.seq)):
    if residue == "-":
        continue
    position += 1
    column_of[position] = column + 1
q7 = str(next(SeqIO.parse("data/reference/Q7WVY1.fasta", "fasta")).seq)
parser = PDBParser(QUIET=True)
structure = parser.get_structure("3cia", "data/pdb/3cia.pdb")
model = structure[0]
for chain_id in [chain.id for chain in model if chain.id != "A"]:
    model.detach_child(chain_id)
chain = model["A"]
for residue in list(chain):
    keep_zinc = residue.resname.strip() == "ZN" and residue.id[1] == 701
    if residue.id[0] != " " and not keep_zinc:
        chain.detach_child(residue.id)
observed = {}
for residue in chain:
    if residue.id[0] != " ":
        continue
    if residue.resname not in THREE:
        raise SystemExit(f"unexpected residue {residue.resname}")
    author = int(residue.id[1])
    if q7[author - 1] != THREE[residue.resname]:
        raise SystemExit(f"PDB sequence mismatch at {author}")
    observed[author] = residue
ShrakeRupley(probe_radius=1.40, n_points=480).compute(model, level="R")
positions = np.array(sorted(observed), dtype=int)
reps = np.vstack([representative_atom(observed[int(position)]) for position in positions])
separation = np.abs(positions[:, None] - positions[None, :])
pair_distance = np.linalg.norm(reps[:, None, :] - reps[None, :, :], axis=2)
contact = (separation >= 6) & (pair_distance <= 8.0)
np.fill_diagonal(contact, False)
n_contacts = contact.sum(axis=1).astype(int)
functional_coords = np.vstack([heavy_coords(observed[position]) for position in FUNCTIONAL])
rows = []
for index, position in enumerate(positions):
    residue = observed[int(position)]
    coords = heavy_coords(residue)
    delta = coords[:, None, :] - functional_coords[None, :, :]
    rows.append({
        "uniprot_position": int(position),
        "reference_residue": q7[int(position) - 1],
        "msa_column_1based": column_of[int(position)],
        "sasa_A2": float(residue.sasa),
        "rsa": float(residue.sasa) / MAX_ASA[q7[int(position) - 1]],
        "n_contacts": int(n_contacts[index]),
        "min_heavy_atom_distance_to_functional_A": float(np.linalg.norm(delta, axis=2).min()),
        "cath_id": domain_of(int(position)),
        "functional_residue": int(position) in FUNCTIONAL,
    })
sites = pd.DataFrame(rows)
sites.to_csv(OUT / "structural_sites.csv", index=False)
